# CS394 Module 6

In this chapter, we aim to build a database for machine learning training.

First, I plan to create a dataset based on [this dataset](https://github.com/iulibdcs/Wright-American-Fiction) that I found on GitHub. This dataset consists of approximately 2,880 novels from 1851 to 1875, organized in XML format. I chose this dataset because the XML format makes it easy to extract just the main text of the novels as I desire. According to the description in the Git repo, it is in the public domain, so there should be no issues with using it.

[Database Creation Pipeline]
- Extract the main text of the novels from the XML files.
- Request the LLM to write a lorebook and an outline of the novel based on the main text.
- Have the LLM create Jinja templates for various novel-related requests (e.g., a request to write Chapter 1 of the novel based on the lorebook and outline), and then combine these with the lorebook and outline created earlier to generate the final prompts.

In [1]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

### Extract only the novel data from the Body in the database.

In [2]:
# import os
# import xml.etree.ElementTree as ET

# def extract_text_per_paragraph(input_folder, output_folder):
#     if not os.path.exists(output_folder):
#         os.makedirs(output_folder)

#     namespaces = {'tei': 'http://www.tei-c.org/ns/1.0'}

#     for filename in os.listdir(input_folder):
#         if filename.endswith(".xml"):
#             file_path = os.path.join(input_folder, filename)
#             output_filename = filename.replace('.xml', '.txt')
#             output_path = os.path.join(output_folder, output_filename)

#             try:
#                 tree = ET.parse(file_path)
#                 root = tree.getroot()
#                 body_element = root.find('.//tei:body', namespaces)

#                 if body_element is not None:
#                     extracted_lines = []
                    
#                     for elem in body_element.iter():
#                         tag_name = elem.tag.split('}')[-1]
                        
#                         if tag_name in ['head', 'p']:
#                             raw_text = "".join(elem.itertext())
#                             single_line_text = " ".join(raw_text.split())
                            
#                             if single_line_text:
#                                 extracted_lines.append(single_line_text)

#                     final_text = "\n\n".join(extracted_lines)

#                     with open(output_path, 'w', encoding='utf-8') as f:
#                         f.write(final_text)
                    
#                     print(f"Processed: {filename} -> {output_filename}")
#                 else:
#                     print(f"Warning: No <body> tag found in {filename}.")

#             except Exception as e:
#                 print(f"Error processing {filename}: {e}")

# INPUT_DIR = '../database/wright_tei'
# OUTPUT_DIR = '../database/wright_body_only'

# extract_text_per_paragraph(INPUT_DIR, OUTPUT_DIR)
# print("Extraction completed.")

### Setup pydantic schema for structured output

In [3]:
from typing import Annotated
from pydantic import BaseModel, Field

class ObjectInfo(BaseModel):
    name: Annotated[str, Field(description="Name of the object.")]
    description: Annotated[str, Field(description="Short description of the object. This should be concise and to the point.")]
    content: Annotated[str, Field(description="Detailed information about the object. This can include various attributes, history, and any relevant details that provide a comprehensive understanding of the object.")]

class ChapterInfo(BaseModel):
    name: Annotated[str, Field(description="Name of the chapter.")]
    description: Annotated[str, Field(description="Short description of the chapter.")]
    content: Annotated[str, Field(description="Detailed information about the chapter.")]
    text_from: Annotated[str, Field(description="The starting sentence of the chapter.")]
    text_to: Annotated[str, Field(description="The ending sentence of the chapter.")]

class ActInfo(BaseModel):
    name: Annotated[str, Field(description="Name of the act.")]
    description: Annotated[str, Field(description="Short description of the act.")]
    content: Annotated[str, Field(description="Detailed information about the act.")]
    chapters: list[ChapterInfo]

class OutlineInfo(BaseModel):
    name: Annotated[str, Field(description="Name of the outline.")]
    description: Annotated[str, Field(description="Short description of the outline.")]
    content: Annotated[str, Field(description="Detailed information about the outline.")]
    acts: list[ActInfo]

class NovelAnalysisResponse(BaseModel):
    genre: Annotated[str, Field(description="Genre of the novel.")]
    logline: Annotated[str, Field(description="Logline of the novel.")]
    characters: Annotated[list[ObjectInfo], Field(description="Characters in the novel.")]
    locations: Annotated[list[ObjectInfo], Field(description="Locations in the novel.")]
    organizations: Annotated[list[ObjectInfo], Field(description="Organizations in the novel.")]
    lorebooks: Annotated[list[ObjectInfo], Field(description="Lores, miscellaneous informations in the novel.")]

    outline: OutlineInfo

### Write code for generate output

#### Test Code

In [ ]:
# import os
# from openai import OpenAI

# def run_llm(system_prompt: str, user_prompt: str, response_type) -> BaseModel:
#     client = OpenAI(
#         # base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
#         api_key=os.environ.get("OPENAI_API_KEY"),
#     )

#     response = client.chat.completions.parse(
#         model="gpt-5-mini",
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user", "content": user_prompt}
#         ],
#         response_format=response_type,
#     )

#     return response.choices[0].message.parsed # type: ignore


In [5]:
system_prompt = (
"""# System

You are an analyst who analyzes novels. You are known as the most highly skilled among analysts who systematically categorize novels. You are someone who perfectly analyzes any novel according to the format without omitting any content.

## Guidelines
The following elements are required for the analysis.
- genre: The genre of this novel. There can be more than one genre, and please separate them with commas.
- logline: Write a logline for this novel consisting of one or two sentences.
- characters: Write down all the main characters appearing in this novel. At this time, you must record in detail the character's appearance, manner of speech, habits, personality, strengths, weaknesses, values, family/friends, past history, current goals, main motives, conflict types, and hidden secrets.
- locations: Write down all the major locations appearing in this novel. At this time, you must record in detail the location's position, appearance, internal structure, atmosphere, and main gimmicks.
- organizations: Write down the major organizations, countries, etc., appearing in this novel. Please record in detail the organizations' values, origins, members, relationships with surrounding organizations, and current goals.
- outline: In the outline, you must separate each novel into acts and chapters, and organize and write the content of each. Even if the chapter and act are not explicitly separated in this novel, you must analyze this novel and appropriately divide it into acts and chapters. In the chapter, the text_from and text_to fields must contain the first sentence of the first paragraph and the last sentence of the last paragraph of each chapter you have categorized.

### Object structure
Each object basically has name, description, and content fields. You must write the name of the object in the name field, briefly summarize the object in one or two sentences in the description field, and write the full-scale analysis content in the content field."""
)

In [ ]:
# user_prompt = ""

# with open("../database/wright_body_only/VAC5518.txt", 'r', encoding='utf-8') as f:
#     user_prompt = f.read()

In [7]:
import os
from openai import OpenAI

def use_openai(system_prompt: str, user_prompt: str, response_type) -> BaseModel:
    client = OpenAI(
        api_key=os.environ.get("OPENAI_API_KEY"),
    )

    response = client.chat.completions.parse(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format=response_type,
    )

    return response.choices[0].message.parsed # type: ignore


In [8]:
import os
from openai import OpenAI

def use_gemini(system_prompt: str, user_prompt: str, response_type) -> BaseModel:
    client = OpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=os.environ.get("GEMINI_API_KEY"),
    )

    response = client.chat.completions.parse(
        model="gemini-3-flash-preview",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format=response_type,
    )

    return response.choices[0].message.parsed # type: ignore


### Dataset generation functions

In [ ]:
# import os
# import json
# import time
# from pathlib import Path
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from tqdm import tqdm

# def process_single_file(file_path, output_folder, system_prompt, schema, llm_function):
#     with open(file_path, 'r', encoding='utf-8') as f:
#         user_prompt = f.read()

#     filename = Path(file_path).stem
#     output_path = os.path.join(output_folder, f"{filename}.json")

#     MAX_RETRIES = 10
#     retry_count = 0
#     while retry_count < MAX_RETRIES:
#         try:
#             data = llm_function(system_prompt, user_prompt, schema)

#             data_dict = data if isinstance(data, dict) else data.dict() 

#             with open(output_path, 'w', encoding='utf-8') as f:
#                 json.dump(data_dict, f, ensure_ascii=False, indent=4)

#             return f"Success: {filename}.json created."
            
#         except Exception as e:
#             if "429" in str(e):
#                 print(f"Error processing {file_path}: {str(e)}")
#                 print(f"Retrying in 15 seconds... (Attempt {retry_count + 1}/{MAX_RETRIES})")
#                 time.sleep(15)
#             else:
#                 return f"Error processing {file_path}: {str(e)}"
#             retry_count += 1

# def process_folder_in_parallel(input_folder, output_folder, system_prompt, schema, max_workers=5):
#     os.makedirs(output_folder, exist_ok=True)

#     input_dir = Path(input_folder)
#     all_txt_files = list(input_dir.glob("*.txt"))
    
#     if not all_txt_files:
#         print("No text files found in the input directory.")
#         return
    
#     txt_files_to_process = []
#     for file_path in all_txt_files:
#         filename = file_path.stem
#         output_path = os.path.join(output_folder, f"{filename}.json")
#         if not os.path.exists(output_path):
#             txt_files_to_process.append(file_path)

#     skipped_count = len(all_txt_files) - len(txt_files_to_process)
#     print(f"Total files: {len(all_txt_files)}. Skipped: {skipped_count}. Remaining to process: {len(txt_files_to_process)}.")

#     if not txt_files_to_process:
#         print("All files have already been processed.")
#         return

#     print(f"Processing {len(txt_files_to_process)} files in parallel using {max_workers} workers...\n")
    
#     half_index = len(txt_files_to_process) // 2
#     gpt_files = txt_files_to_process[:half_index]
#     gemini_files = txt_files_to_process[half_index:]

#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         futures = {}
        
#         for file_path in gpt_files:
#             future = executor.submit(process_single_file, file_path, output_folder, system_prompt, schema, use_openai)
#             futures[future] = file_path
            
#         for file_path in gemini_files:
#             future = executor.submit(process_single_file, file_path, output_folder, system_prompt, schema, use_gemini)
#             futures[future] = file_path

#         for future in tqdm(as_completed(futures), total=len(txt_files_to_process), desc="Processing files"):
#             result = future.result()
#             tqdm.write(result)

#     print("\nAll processes completed.")

In [10]:
# Test
# process_single_file('../database/wright_body_only/VAC5518.txt', "../database/wright_analysis", system_prompt, NovelAnalysisResponse)

In [ ]:
# INPUT_DIR = "../database/wright_body_only"
# OUTPUT_DIR = "../database/wright_analysis"
# N_WORKERS = 24

# process_folder_in_parallel(
#     input_folder=INPUT_DIR,
#     output_folder=OUTPUT_DIR,
#     system_prompt=system_prompt,
#     schema=NovelAnalysisResponse,
#     max_workers=N_WORKERS
# )

Total files: 2876. Skipped: 197. Remaining to process: 2679.
Processing 2679 files in parallel using 24 workers...



Processing files:   0%|          | 0/2679 [00:00<?, ?it/s]

Error processing ../database/wright_body_only/VAC5619.txt: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5-mini in organization org-VQzCGvXyvmGFXqgFQo44JpzT on tokens per min (TPM): Limit 500000, Used 500000, Requested 65826. Please try again in 7.899s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
Retrying in 15 seconds... (Attempt 1/10)
Error processing ../database/wright_body_only/VAC5621.txt: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5-mini in organization org-VQzCGvXyvmGFXqgFQo44JpzT on tokens per min (TPM): Limit 500000, Used 456602, Requested 74696. Please try again in 3.755s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
Retrying in 15 seconds... (Attempt 1/10)
Error processing ../database/wright_body_only/VAC5620.txt: Error code: 429 - {'error': {'messa

Processing files:   0%|          | 1/2679 [03:19<148:31:37, 199.66s/it]

Error processing ../database/wright_body_only/VAC5529.txt: Error code: 400 - {'error': {'message': 'Input tokens exceed the configured limit of 272000 tokens. Your messages resulted in 297821 tokens. Please reduce the length of the messages.', 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}


/tmp/ipykernel_49517/400749765.py:21: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  data_dict = data if isinstance(data, dict) else data.dict()
Processing files:   0%|          | 2/2679 [03:33<66:59:18, 90.09s/it]  

... (166 outputs left out)

Success: VAC5806.json created.


Processing files:   3%|▎         | 87/2679 [29:34<14:40:58, 20.39s/it]


Error processing ../database/wright_body_only/VAC5826.txt: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5-mini in organization org-VQzCGvXyvmGFXqgFQo44JpzT on tokens per min (TPM): Limit 500000, Used 360143, Requested 152180. Please try again in 1.478s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
Retrying in 15 seconds... (Attempt 1/10)
Error processing ../database/wright_body_only/VAC5828.txt: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5-mini in organization org-VQzCGvXyvmGFXqgFQo44JpzT on tokens per min (TPM): Limit 500000, Used 500000, Requested 155318. Please try again in 18.638s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
Retrying in 15 seconds... (Attempt 1/10)
Error processing ../database/wright_body_only/VAC5767.txt: Error code: 429 - {'error': {'me

TypeError: write() argument must be str, not <class 'NoneType'>

Running 24 items with GPT-5.2 cost me $14. I feel like I'm going to go bankrupt at this rate, so I’m going to switch models. I used gpt-5-mini and gemini-3-flesh together, but it still cost more than $20, so I don't think I can create any more data. An error occurred in the middle, so I couldn't finish it properly, but since I can't let any more of my money drain away, I'll just stop here for now. I've commented out the code to prevent the disaster of accidentally sending requests again.

### Merge the content with the original novel based on the generated analysis.

In [ ]:
import os
import json
import uuid
import re

ANALYSIS_DIR = "../database/wright_analysis"
TEXT_DIR = "../database/wright_body_only"
OUTPUT_DIR = "../database/wright_final"

def generate_id():
    return uuid.uuid4().hex

def find_flexible_index(full_text, target_text, start_pos=0, is_end=False):
    """
    Find the index of target_text in full_text starting from start_pos. 
    If is_end is True, find the end index of the match; otherwise, find the start index.
    """
    if not target_text.strip():
        return -1

    words = target_text.split()
    regex_pattern = r'\s+'.join(re.escape(word) for word in words)
    
    match = re.search(regex_pattern, full_text[start_pos:])
    
    if match:
        if is_end:
            return start_pos + match.end()
        else:
            return start_pos + match.start()
            
    return -1

def find_nearest_paragraph_break(full_text, ideal_idx, A, B):
    """
    Find the nearest paragraph break (double newline) to the ideal index within the range [A, B].
    """
    breaks = []
    pos = A
    while True:
        pos = full_text.find('\n\n', pos, B)
        if pos == -1:
            break
        breaks.append(pos)
        pos += 2
        
    if not breaks:
        return ideal_idx
        
    closest_break = min(breaks, key=lambda x: abs(x - ideal_idx))
    return closest_break

def format_objects(obj_list):
    """
    Helper function to format characters, locations, organizations, and lorebooks.
    """
    return [
        {
            "id": generate_id(),
            "name": obj.get("name", ""),
            "description": obj.get("description", ""),
            "content": obj.get("content", "")
        }
        for obj in obj_list
    ]

def process_novels():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    for filename in os.listdir(ANALYSIS_DIR):
        if not filename.endswith(".json"):
            continue
            
        base_name = os.path.splitext(filename)[0]
        json_path = os.path.join(ANALYSIS_DIR, filename)
        txt_path = os.path.join(TEXT_DIR, f"{base_name}.txt")
        
        if not os.path.exists(txt_path):
            continue
            
        print(f"Processing: {base_name}")
        
        with open(json_path, 'r', encoding='utf-8') as jf:
            analysis_data = json.load(jf)
            
        with open(txt_path, 'r', encoding='utf-8') as tf:
            full_text = tf.read()
            
        flat_chapters = []
        original_acts = analysis_data.get("outline", {}).get("acts", [])
        
        for act in original_acts:
            for chapter in act.get("chapters", []):
                flat_chapters.append({
                    "id": generate_id(),
                    "text_from": chapter.get("text_from", ""),
                    "text_to": chapter.get("text_to", ""),
                    "original_data": chapter,
                    "s": -1,
                    "e": -1
                })
        
        ##### Chapter boundary detection #####
        # Pipeline 1: Try to find both boundaries using the provided text snippets
        current_pos = 0
        for ch in flat_chapters:
            s_idx = find_flexible_index(full_text, ch["text_from"], start_pos=current_pos, is_end=False)
            if s_idx != -1:
                current_pos = s_idx
            ch["s"] = s_idx
            
            e_idx = find_flexible_index(full_text, ch["text_to"], start_pos=current_pos, is_end=True)
            if e_idx != -1:
                current_pos = e_idx
            ch["e"] = e_idx

        # Pipeline 2: Fill in missing boundaries
        for i, ch in enumerate(flat_chapters):
            if ch["s"] == -1 and ch["e"] != -1:
                nearest_e = 0
                for j in range(i - 1, -1, -1):
                    if flat_chapters[j]["e"] != -1:
                        nearest_e = flat_chapters[j]["e"]
                        break
                ch["s"] = nearest_e
                
            elif ch["s"] != -1 and ch["e"] == -1:
                nearest_s = len(full_text)
                for j in range(i + 1, len(flat_chapters)):
                    if flat_chapters[j]["s"] != -1:
                        nearest_s = flat_chapters[j]["s"]
                        break
                ch["e"] = nearest_s

        # Pipeline 3: If there are consecutive chapters with missing boundaries, split the gap evenly
        i = 0
        while i < len(flat_chapters):
            if flat_chapters[i]["s"] == -1 and flat_chapters[i]["e"] == -1:
                j = i
                while j < len(flat_chapters) and flat_chapters[j]["s"] == -1 and flat_chapters[j]["e"] == -1:
                    j += 1
                
                n_missing = j - i
                
                print(f"Found {n_missing + 1} consecutive chapters with missing boundaries from index {i} to {j}.")
                A = 0 if i == 0 else flat_chapters[i-1]["e"]
                B = len(full_text) if j == len(flat_chapters) else flat_chapters[j]["s"]
                
                if B < A:
                    B = A
                
                if n_missing == 1:
                    flat_chapters[i]["s"] = A
                    flat_chapters[i]["e"] = B
                else:
                    chunk_size = (B - A) / n_missing
                    split_points = [A]
                    
                    for k in range(1, n_missing):
                        ideal_idx = int(A + k * chunk_size)
                        actual_idx = find_nearest_paragraph_break(full_text, ideal_idx, A, B)
                        split_points.append(actual_idx)
                        
                    split_points.append(B)
                    
                    for k in range(n_missing):
                        curr = i + k
                        flat_chapters[curr]["s"] = split_points[k]
                        flat_chapters[curr]["e"] = split_points[k+1]
                
                i = j 
            else:
                i += 1
                
        extracted_texts = {}
        for ch in flat_chapters:
            s = ch["s"]
            e = ch["e"]
            extracted_texts[ch["id"]] = full_text[s:e].strip()

        new_data = {
            "genre": analysis_data.get("genre", ""),
            "logline": analysis_data.get("logline", ""),
            "characters": format_objects(analysis_data.get("characters", [])),
            "locations": format_objects(analysis_data.get("locations", [])),
            "organizations": format_objects(analysis_data.get("organizations", [])),
            "lorebooks": format_objects(analysis_data.get("lorebooks", [])),
            "outline": {
                "id": generate_id(),
                "name": analysis_data.get("outline", {}).get("name", ""),
                "description": analysis_data.get("outline", {}).get("description", ""),
                "content": analysis_data.get("outline", {}).get("content", ""),
                "acts": []
            },
            "manuscript": {
                "chapters": []
            },
            "guidelines": {
                "id": generate_id(),
                "authorNote": ""
            }
        }
        
        flat_idx = 0
        for act in original_acts:
            new_act = {
                "id": generate_id(),
                "name": act.get("name", ""),
                "description": act.get("description", ""),
                "content": act.get("content", ""),
                "chapters": []
            }
            
            for chapter in act.get("chapters", []):
                ch_info = flat_chapters[flat_idx]
                ch_id = ch_info["id"]
                
                new_act["chapters"].append({
                    "id": ch_id,
                    "name": chapter.get("name", ""),
                    "description": chapter.get("description", ""),
                    "content": chapter.get("content", ""),
                    "text_from": chapter.get("text_from", ""),
                    "text_to": chapter.get("text_to", "")
                })
                
                new_data["manuscript"]["chapters"].append({
                    "id": ch_id,
                    "name": chapter.get("name", ""),
                    "chapter_text": extracted_texts[ch_id]
                })
                flat_idx += 1
                
            new_data["outline"]["acts"].append(new_act)
            
        output_path = os.path.join(OUTPUT_DIR, filename)
        with open(output_path, 'w', encoding='utf-8') as outf:
            json.dump(new_data, outf, ensure_ascii=False, indent=2)


process_novels()
print("Process Finished.")

Processing: VAC5518
Found 3 consecutive chapters with missing boundaries from index 3 to 5.
Found 6 consecutive chapters with missing boundaries from index 6 to 11.
Found 3 consecutive chapters with missing boundaries from index 13 to 15.
Found 2 consecutive chapters with missing boundaries from index 17 to 18.
Processing: VAC5519
Found 11 consecutive chapters with missing boundaries from index 10 to 20.
Found 9 consecutive chapters with missing boundaries from index 21 to 29.
Found 6 consecutive chapters with missing boundaries from index 30 to 35.
Processing: VAC5520
Processing: VAC5521
Processing: VAC5522
Found 2 consecutive chapters with missing boundaries from index 29 to 30.
Processing: VAC5523
Processing: VAC5524
Found 2 consecutive chapters with missing boundaries from index 15 to 16.
Found 3 consecutive chapters with missing boundaries from index 18 to 20.
Processing: VAC5525
Processing: VAC5526
Found 3 consecutive chapters with missing boundaries from index 6 to 8.
Found 2 co

### Create a conversation template based on the generated Jinja template.

I created various Jinja template files for generating conversation templates using Opus 4.6, and the prompts used to create these templates are stored in the template_prompt.md file.

In [20]:
from pathlib import Path
import json
from jinja2 import Environment, FileSystemLoader

TEMPLATE_DIR = Path("templates")
DATA_DIR = Path("../database/wright_combined")
OUT_DIR = Path("../database/wright_rendered")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def combine(a, b):
    d = dict(a or {})
    d.update(b or {})
    return d

env = Environment(loader=FileSystemLoader(str(TEMPLATE_DIR)))
env.filters["combine"] = combine

template_names = [
    t for t in sorted(env.list_templates())
    if re.match(r"^(0[1-9]|1[0-9]|2[0-5])_.*\.jinja2$", t)
]
targets = sorted(DATA_DIR.glob("*.json"))

for tname in template_names:
    template = env.get_template(tname)
    out_path = OUT_DIR / f"{Path(tname).stem}.jsonl"

    with out_path.open("w", encoding="utf-8") as fout:
        for jf in targets:
            with jf.open("r", encoding="utf-8") as f:
                record = json.load(f)

            rendered = template.render(**record).strip()
            if not rendered:
                continue

            for line in rendered.splitlines():
                line = line.strip()
                if not line:
                    continue
                fout.write(json.dumps(json.loads(line), ensure_ascii=False) + "\n")

    print(f"[OK] {tname} -> {out_path}")


[OK] 01_setup_basic_info.jinja2 -> ../database/wright_rendered/01_setup_basic_info.jsonl
[OK] 02_create_world_elements.jinja2 -> ../database/wright_rendered/02_create_world_elements.jsonl
[OK] 03_create_outline.jinja2 -> ../database/wright_rendered/03_create_outline.jsonl
[OK] 04_write_manuscript.jinja2 -> ../database/wright_rendered/04_write_manuscript.jsonl
[OK] 05_edit_and_revise.jinja2 -> ../database/wright_rendered/05_edit_and_revise.jsonl
[OK] 06_retrieval_and_search.jinja2 -> ../database/wright_rendered/06_retrieval_and_search.jsonl
[OK] 07_full_workflow.jinja2 -> ../database/wright_rendered/07_full_workflow.jsonl
[OK] 08_character_relationship_mapping.jinja2 -> ../database/wright_rendered/08_character_relationship_mapping.jsonl
[OK] 09_continuity_check.jinja2 -> ../database/wright_rendered/09_continuity_check.jsonl
[OK] 10_chapter_continuation.jinja2 -> ../database/wright_rendered/10_chapter_continuation.jsonl
[OK] 11_style_tone_adjustment.jinja2 -> ../database/wright_rendered/

In [27]:
import json
import random
from pathlib import Path

def split_to_jsonl_only(
    rendered_dir="../database/wright_rendered",
    out_dir="../database/wright_rendered_splits",
    ratio=(9, 1.0, 0.001),
    seed=42
):
    rendered_dir = Path(rendered_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    bundles = []
    for fp in sorted(rendered_dir.glob("*.jsonl")):
        with fp.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    bundles.append(json.loads(line))

    rng = random.Random(seed)
    rng.shuffle(bundles)

    n = len(bundles)
    total = sum(ratio)
    n_train = int(n * ratio[0] / total)
    n_validate = int(n * ratio[1] / total)

    train = bundles[:n_train]
    validate = bundles[n_train:n_train + n_validate]
    test = bundles[n_train + n_validate:]

    def write_jsonl(path, rows):
        with path.open("w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    write_jsonl(out_dir / "train.jsonl", train)
    write_jsonl(out_dir / "validate.jsonl", validate)
    write_jsonl(out_dir / "test.jsonl", test)

    return {
        "total": n,
        "train": len(train),
        "validate": len(validate),
        "test": len(test),
    }


In [28]:
meta = split_to_jsonl_only()
print(meta)

{'total': 118496, 'train': 106635, 'validate': 11848, 'test': 13}


Ultimately, I set the numbers to 100k for the training set, approximately 10k for the validation set, and about 10 for the test set.